# Лабораторная работа №7
## Анализ текста

**Выполнил:** Брискиндов Олег Романович, группа **P3124**


Перед запуском данного борда необходимо сделать следующее:

0. Сделать копию борда через меню "Файл" / "File".
1. Выбрать пункт "Среда выполнения" или "Runtime" и выбрать GPU в качестве аппаратного ускорителя в пункте меню "Сменить среду выполнения".

# Инструменты для работы с языком

... или зачем нужна предобработка.

Раньше мы смотрели на светлую сторону анализа данных - построение моделей. Теперь попробуем глубже посмотреть на часть про предобработку данных. Задача предобработки особенно актуальна, если мы имеем дело с текстами.

## Задача: классификация твитов по тональности

У нас есть выборка из твитов. Нам известна эмоциональная окраска каждого твита из выборки: положительная или отрицательная. Задача состоит в построении модели, которая по тексту твита предсказывает его эмоциональную окраску.

Классификацию по тональности используют в рекомендательных системах, чтобы понять, понравилось ли людям кафе, кино, etc.

Скачиваем выборку ([источник](http://study.mokoron.com/)): [положительные](https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/positive.csv), [отрицательные](https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/negative.csv).

In [11]:
!wget https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/positive.csv
!wget https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/negative.csv

zsh:1: command not found: wget
zsh:1: command not found: wget


In [12]:
import pandas as pd  # библиотека для удобной работы с датафреймами
import numpy as np   # библиотека для удобной работы со списками и матрицами

# библиотека, где реализованы основные алгоритмы машинного обучения
from sklearn.metrics import *
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [13]:
!head positive.csv

"408906692374446080";"1386325927";"pleease_shut_up";"@first_timee хоть я и школота, но поверь, у нас то же самое :D общество профилирующий предмет типа)";"1";"0";"0";"0";"7569";"62";"61";"0"
"408906692693221377";"1386325927";"alinakirpicheva";"Да, все-таки он немного похож на него. Но мой мальчик все равно лучше:D";"1";"0";"0";"0";"11825";"59";"31";"2"
"408906695083954177";"1386325927";"EvgeshaRe";"RT @KatiaCheh: Ну ты идиотка) я испугалась за тебя!!!";"1";"0";"1";"0";"1273";"26";"27";"0"
"408906695356973056";"1386325927";"ikonnikova_21";"RT @digger2912: ""Кто то в углу сидит и погибает от голода, а мы ещё 2 порции взяли, хотя уже и так жрать не хотим"" :DD http://t.co/GqG6iuE2…";"1";"0";"1";"0";"1549";"19";"17";"0"
"408906761416867842";"1386325943";"JumpyAlex";"@irina_dyshkant Вот что значит страшилка :D
Но блин,посмотрев все части,у тебя создастся ощущение,что авторы курили что-то :D";"1";"0";"0";"0";"597";"16";"23";"1"
"408906761769598976";"1386325943";"JustinB94262583";"ну любишь и

In [14]:
!tail negative.csv

"425137932283158528";"1390195756";"pazyfevesity";"RT @qelasocadij: Скажите пожалуйста, как у человека может быть 1000 одноклассников? O_o";"-1";"0";"1";"0";"2168";"171";"132";"0"
"425137934443233281";"1390195756";"Sonya_Star_14";"У нас физ ра на улице
Пака линт:(
Через 45 минут приду пхжааххв";"-1";"0";"0";"0";"12722";"638";"567";"2"
"425138035089358848";"1390195780";"evalesana";"Нас сегодня отказались принять в сад, типа мы плачем(( #королев пойду ругаться сейчас, по крайне мере выяснять, что за фигня";"-1";"0";"0";"0";"4101";"166";"151";"6"
"425138243257253888";"1390195830";"Yanch_96";"Но не каждый хочет что то исправлять:( http://t.co/QNODDQzuZ7";"-1";"0";"0";"0";"1138";"32";"46";"0"
"425138339503943682";"1390195853";"tkit_on";"скучаю так :-( только @taaannyaaa вправляет мозги, но я все равно скучаю";"-1";"0";"0";"0";"4822";"38";"32";"0"
"425138437684215808";"1390195876";"ckooker1";"Вот и в школу, в говно это идти уже надо(";"-1";"0";"0";"1";"165";"13";"16";"0"
"425138490452344832";

Откроем файлы и создадим массив из текстов и правильных меток для твитов. Сначала идут положительные твиты, потом отрицательные.

In [15]:
# загружаем положительные твиты
# TODO #2
positive = pd.read_csv('positive.csv', sep=';', usecols=[3], names=['text'])
positive['label'] = ['positive'] * len(positive)

# загружаем отрицательные твиты
# TODO #3
negative = pd.read_csv('negative.csv', sep=';', usecols=[3], names=['text'])
negative['label'] = ['negative'] * len(negative)

# соединяем вместе
# TODO #4
df = pd.concat([positive, negative])

In [16]:
df

,text,label
0,"@first_timee хоть я и школота, но поверь, у на...",positive
1,"Да, все-таки он немного похож на него. Но мой ...",positive
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,positive
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",positive
4,@irina_dyshkant Вот что значит страшилка :D\nН...,positive
...,...,...
111918,Но не каждый хочет что то исправлять:( http://...,negative
111919,скучаю так :-( только @taaannyaaa вправляет мо...,negative
111920,"Вот и в школу, в говно это идти уже надо(",negative
111921,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",negative


Посмотрим на полученные данные:

In [17]:
df.sample(5, random_state=40)

,text,label
15931,RT @Blawar_1337: Теперь у нас с @Wake_UA появи...,positive
59532,с днём рождения зайка*))) ухх погуляем мы сего...,positive
47185,RT @Shumkova0406199: @ann_safina Вов вов вов А...,negative
42002,"Надо выдернуть звуковую дорожку из ""Доктора Ка...",positive
109035,@_hassliebe_ может все таки на этой неделе вер...,negative


Разбиваем данные на обучающую и тестовую выборки с помощью функции `train_test_split()` из **sklearn**:

In [18]:
x_train, x_test, y_train, y_test = train_test_split(df.text, df.label, random_state=0)

print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

(170125,) (56709,) (170125,) (56709,)


In [19]:
y_train[:10]

114739    positive
104839    positive
8198      positive
65307     positive
15893     positive
85064     negative
97884     positive
50039     negative
90284     negative
15971     negative
Name: label, dtype: object

In [20]:
y_train.value_counts()

label
positive    86131
negative    83994
Name: count, dtype: int64

## Baseline: классификация необработанных n-грамм

* Сейчас мы попробуем получить преобразование предложений в численный вектор, с которым может работать стандартный алгоритм машинного обучения, такой как логистическая регрессия.
* Для этого нам понадобится познакомиться с понятием n-gram - самых мелких элементов предложения, с которыми можно работать.
* Подсчитав количество этих n-грамм в предложениях, мы получим искомые численные представления.

## Что такое n-граммы:

Самые мелкие структуры языка, с которыми мы работаем, называются **n-граммами**. У n-граммы есть параметр n - количество слов, которые попадают в такое представление текста.
* Если n = 1 - то мы смотрим на то, сколько раз каждое слово встретилось в тексте. Получаем _униграммы_
* Если n = 2 - то мы смотрим на то, сколько раз каждая пара подряд идущих слов, встретилась в тексте. Получаем _биграммы_

**Функция** для работы с n-граммами реализована в библиотке **nltk** (Natural Language ToolKit), импортируем эту функцию:

In [21]:
from nltk import ngrams

Прежде чем получать n-граммы, нужно разделить предложение на отдельные слова. Для этого используем метод `split()`.

In [22]:
sentence = 'Если б мне платили каждый раз'.split()
sentence

['Если', 'б', 'мне', 'платили', 'каждый', 'раз']

Чтобы получить n-грамму для такой последовательности, используем функцию `ngrams()`. Чтобы полученный объект отобразить, делаем из него `list`.

In [23]:
list(ngrams(sentence, 1))  # униграммы

[('Если',), ('б',), ('мне',), ('платили',), ('каждый',), ('раз',)]

Аналогично мы можем получить биграммы - для этого заменяем параметр **n** в функции **ngrams** с 1 на 2.

In [24]:
list(ngrams(sentence, 2))  # биграммы

[('Если', 'б'),
 ('б', 'мне'),
 ('мне', 'платили'),
 ('платили', 'каждый'),
 ('каждый', 'раз')]

In [25]:
list(ngrams(sentence, 3))  # триграммы

[('Если', 'б', 'мне'),
 ('б', 'мне', 'платили'),
 ('мне', 'платили', 'каждый'),
 ('платили', 'каждый', 'раз')]

In [26]:
list(ngrams(sentence, 5))  # ... пентаграммы?

[('Если', 'б', 'мне', 'платили', 'каждый'),
 ('б', 'мне', 'платили', 'каждый', 'раз')]

### Векторизаторы

Векторизатор преобразует слово или набор слов в числовой вектор, понятный алгоритму машинного обучения, который привык работать с числовыми табличными данными.

На начальном этапе нам будет достаточно тех инструментов, которые уже есть в знакомой нам библиотеке **sklearn**.

In [27]:
from sklearn.linear_model import LogisticRegression  # можно заменить на любимый классификатор
from sklearn.feature_extraction.text import CountVectorizer  # модель "мешка слов"

Самый простой способ извлечь признаки из текстовых данных — векторизаторы: `CountVectorizer` и `TfidfVectorizer`.

Объект `CountVectorizer`:
* строит для каждого документа вектор размерности `n`, где `n` — количество слов или n-грамм во всём корпусе;
* заполняет каждый i-тый элемент количеством вхождений слова в данный документ.

На рисунке пример векторизации для униграмм, но можно использовать любые n-граммы. Для этого у объекта `CountVectorizer()` есть параметр **ngram_range**:
- `ngram_range=(1, 1)` — униграммы
- `ngram_range=(3, 3)` — триграммы
- `ngram_range=(1, 3)` — униграммы, биграммы и триграммы.

Инициализируем `CountVectorizer()`, указав в качестве признаков униграммы:

In [28]:
vectorizer = CountVectorizer(ngram_range=(1,1))

После инициализации `vectorizer` можно обучить на наших данных. Используем метод `fit_transform()`: сначала обучаем наш векторизатор, а потом сразу применяем его к нашему набору данных.

In [29]:
# TODO #8
vectorized_x_train = vectorizer.fit_transform(x_train)

Так как результат не зависит от порядка слов в текстах, то говорят, что такая модель представления текстов в виде векторов получается из *гипотезы представления текста как мешка слов*.

В `vectorizer.vocabulary_` лежит словарь, отображение слов в их индексы:

In [30]:
list(vectorizer.vocabulary_.items())[:10]

[('nasstik', 61112),
 ('хостес', 233758),
 ('армянами', 100964),
 ('целуется', 235200),
 ('на', 161276),
 ('посту', 190048),
 ('alinasafary', 11054),
 ('думаю', 128476),
 ('испугались', 141187),
 ('те', 220294)]

В нашей выборке 170125 текстов (твитов), в них встречается ~243к разных слов.

In [31]:
vectorized_x_train.shape

(170125, 243283)

Так как теперь у нас есть **численное представление** и набор входных признаков, то мы можем обучить модель логистической регрессии.

In [32]:
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(vectorized_x_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

С тестовыми данными нужно проделать то же самое, что и с данными для обучения. У нас уже есть обученный векторизатор `vectorizer`, поэтому используем `transform()`.

In [33]:
vectorized_x_test = vectorizer.transform(x_test)

С помощью функции `classification_report()` посмотрим на то, насколько хорошо мы предсказываем тональность твита.

In [34]:
pred = clf.predict(vectorized_x_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.76      0.77      0.76     27929
    positive       0.77      0.76      0.77     28780

    accuracy                           0.77     56709
   macro avg       0.77      0.77      0.77     56709
weighted avg       0.77      0.77      0.77     56709



## Бонус*: триграммы

Попробуем сделать то же самое, используя в качестве признаков триграммы:

In [35]:
# TODO #12

# инициализируем векторайзер
vectorizer_3 = CountVectorizer(ngram_range=(3,3))
# обучаем его и сразу применяем к x_train
vectorized_x_train_3 = vectorizer_3.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(vectorized_x_train_3, y_train)
# применяем обученный векторизатор к тестовым данным
vectorized_x_test_3 = vectorizer_3.transform(x_test)
# получаем предсказания и выводим информацию о качестве
pred = clf.predict(vectorized_x_test_3)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.71      0.47      0.56     27929
    positive       0.61      0.81      0.70     28780

    accuracy                           0.64     56709
   macro avg       0.66      0.64      0.63     56709
weighted avg       0.66      0.64      0.63     56709



**Почему такой разброс?** Триграммы — слишком жёсткое представление: в тестовой выборке встречается множество троек слов, которых не было в обучающей. Классификатор для таких документов почти всегда «угадывает» большинство и сваливается в более частый класс — отсюда низкий recall у negative и высокий у positive.

## Бонус**: TF-IDF векторизация

`TfidfVectorizer` делает то же, что и `CountVectorizer`, но в качестве значений выдает **tf-idf** каждого слова.

Как считается tf-idf:

**TF (term frequency)** — относительная частотность слова в документе:
$$ TF(t,d) = \frac{n_{t}}{\sum_k n_{k}} $$

**IDF (inverse document frequency)** — обратная частота документов, в которых есть это слово:
$$ IDF(t, D) = \log \frac{|D|}{|\{d : t \in d\}|} $$

Перемножаем их:
$$TFIDF(t, d, D) = TF(t,d) \times IDF(t, D)$$

Сакральный смысл: если слово часто встречается в одном документе, но в целом по корпусу встречается в небольшом количестве документов, у него высокий TF-IDF.

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

Действуем аналогично, как с `CountVectorizer()`:

In [37]:
# TODO #13

# инициализируем векторизатор, в качестве признаков используем униграммы..пентаграммы
tfidfvectorizer = TfidfVectorizer(ngram_range=(1,5))
# обучаем его и сразу применяем к x_train
tfidf_vectorized_x_train = tfidfvectorizer.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(tfidf_vectorized_x_train, y_train)
# применяем обученный векторизатор к тестовым данным
tfidf_vectorized_x_test = tfidfvectorizer.transform(x_test)
# получаем предсказания и выводим информацию о качестве
pred = clf.predict(tfidf_vectorized_x_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.73      0.78      0.76     27929
    positive       0.77      0.72      0.75     28780

    accuracy                           0.75     56709
   macro avg       0.75      0.75      0.75     56709
weighted avg       0.75      0.75      0.75     56709



### TF-IDF: пентаграммы

In [38]:
# TODO #14

# инициализируем векторизатор, в качестве переменных используем пентаграммы
tfidf_5 = TfidfVectorizer(ngram_range=(5,5))
# обучаем его и сразу применяем к x_train
tfidf_x_train_5 = tfidf_5.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(tfidf_x_train_5, y_train)
# применяем обученный векторизатор к тестовым данным
tfidf_x_test_5 = tfidf_5.transform(x_test)
# получаем предсказания и выводим информацию о качестве
pred = clf.predict(tfidf_x_test_5)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.95      0.11      0.20     27929
    positive       0.54      0.99      0.70     28780

    accuracy                           0.56     56709
   macro avg       0.74      0.55      0.45     56709
weighted avg       0.74      0.56      0.45     56709



## Токенизация

Токенизировать — значит, поделить текст на части: слова, ключевые слова, фразы, символы и т.д., иными словами **токены**.

Самый наивный способ токенизировать текст — разделить с помощью функции `split()`. Но `split` упускает очень много всего, например, не отделяет пунктуацию от слов. Кроме этого, есть ещё много менее тривиальных проблем, поэтому лучше использовать готовые токенизаторы.

In [39]:
import nltk
from nltk.tokenize import word_tokenize  # готовый токенизатор библиотеки nltk

Чтобы использовать токенизатор `word_tokenize`, нужно сначала скачать данные для nltk о пунктуации и стоп-словах:

In [40]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Error loading punkt_tab: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1028)>
[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1028)>


False

Применим токенизацию:

In [41]:
example = 'Но не каждый хочет что-то исправлять:('
word_tokenize(example)

['Но', 'не', 'каждый', 'хочет', 'что-то', 'исправлять', ':', '(']

Если использовать просто `split()`, то грустный смайлик `:(` не отделяется от слова "исправлять":

In [42]:
example.split()

['Но', 'не', 'каждый', 'хочет', 'что-то', 'исправлять:(']

В nltk вообще есть довольно много токенизаторов:

In [43]:
from nltk import tokenize
dir(tokenize)[:16]

['BlanklineTokenizer',
 'LegalitySyllableTokenizer',
 'LineTokenizer',
 'MWETokenizer',
 'NLTKWordTokenizer',
 'PunktSentenceTokenizer',
 'PunktTokenizer',
 'RegexpTokenizer',
 'ReppTokenizer',
 'SExprTokenizer',
 'SpaceTokenizer',
 'StanfordSegmenter',
 'SyllableTokenizer',
 'TabTokenizer',
 'TextTilingTokenizer',
 'ToktokTokenizer']

Они умеют выдавать индексы в строке для начала и конца каждого слова-токена:

In [44]:
wh_tok = tokenize.WhitespaceTokenizer()
list(wh_tok.span_tokenize(example))

[(0, 2), (3, 5), (6, 12), (13, 18), (19, 25), (26, 38)]

Некоторые токенизаторы ведут себя специфично:

In [45]:
tokenize.TreebankWordTokenizer().tokenize("don't stop me")

['do', "n't", 'stop', 'me']

А некоторые - вообще не для текста на естественном языке:

In [46]:
tokenize.SExprTokenizer().tokenize("(a (b c)) d e (f)")

['(a (b c))', 'd', 'e', '(f)']

**Правильный токенизатор подбирается исходя из требований задачи!**

## Стоп-слова и пунктуация

**Стоп-слова** — это слова, которые часто встречаются практически в любом тексте и ничего интересного не говорят о конкретном документе. Для модели это просто шум.

In [47]:
from nltk.corpus import stopwords

print(stopwords.words('russian'))

['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя', 'впр

*Знаки* пунктуации лучше импортировать из модуля **String**:

In [48]:
from string import punctuation
punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

Объединим стоп-слова и знаки пунктуации вместе и запишем в переменную `noise`:

In [49]:
noise = stopwords.words('russian') + list(punctuation)

Теперь нужно обучать нашу модель с учетом новых знаний про токенизацию и стоп-слова. Для этого мы можем собрать новый векторизатор.

In [50]:
# инициализируем умный векторайзер
# TODO #16 с word_tokenize
smart_tokenizer = CountVectorizer(
    ngram_range=(1, 1),
    tokenizer=word_tokenize,
    stop_words=noise,
)

In [51]:
# TODO #17

# обучаем его и сразу применяем к x_train
smart_x_train = smart_tokenizer.fit_transform(x_train)

# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(smart_x_train, y_train)

# применяем обученный векторайзер к тестовым данным
smart_x_test = smart_tokenizer.transform(x_test)

# получаем предсказания и выводим информацию о качестве
pred = clf.predict(smart_x_test)
print(classification_report(y_test, pred))

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['``'] not in stop_words.
  warnings.warn(


              precision    recall  f1-score   support

    negative       0.76      0.80      0.78     27929
    positive       0.80      0.76      0.78     28780

    accuracy                           0.78     56709
   macro avg       0.78      0.78      0.78     56709
weighted avg       0.78      0.78      0.78     56709



Получилось лучше: accuracy выше, а также заметно подрос recall у негативного класса. Что ещё можно сделать?

## Бонус*: Лемматизация

**Лемматизация** — это сведение разных форм одного слова к начальной форме — **лемме**. Почему это хорошо?
* Естественно рассматривать как отдельный признак каждое *слово*, а не каждую его отдельную форму.
* Некоторые стоп-слова стоят только в начальной форме, и без лемматизации выкидываем мы только её.

Для русского есть хороший лемматизатор pymorphy.

### [Pymorphy](http://pymorphy2.readthedocs.io/en/latest/)
Это модуль на питоне, довольно быстрый и с кучей функций.

In [52]:
!pip install pymorphy3

В pymorphy3 для морфологического анализа слов есть `MorphAnalyzer()`:

In [53]:
from pymorphy3 import MorphAnalyzer
# создадим объект pymorphy3_analyzer импортированного класса

# TODO #18
pymorphy3_analyzer = MorphAnalyzer()

pymorphy3 работает с отдельными словами:

In [54]:
sent = ['Если', 'б', 'мне', 'платили', 'каждый', 'раз']
sent

['Если', 'б', 'мне', 'платили', 'каждый', 'раз']

Лемматизируем слово "платили" из предложения `sent` с помощью метода `parse()`:

In [55]:
ana = pymorphy3_analyzer.parse(sent[3])
ana

[Parse(word='платили', tag=OpencorporaTag('VERB,impf,tran plur,past,indc'), normal_form='платить', score=1.0, methods_stack=((DictionaryAnalyzer(), 'платили', 2471, 10),))]

Выведем его нормальную форму:

In [56]:
ana[0].normal_form

'платить'

## О важности эксплоративного анализа

Но иногда пунктуация бывает и не шумом — главное отталкиваться от задачи. Что будет, если вообще не убирать пунктуацию?

In [57]:
# TODO #19

# инициализируем умный векторайзер - стоп-слова НЕ ИСПОЛЬЗУЕМ!
vectorizer_no_stop = CountVectorizer(ngram_range=(1, 1), tokenizer=word_tokenize)

# обучаем его и сразу применяем к x_train
x_train_no_stop = vectorizer_no_stop.fit_transform(x_train)

# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(x_train_no_stop, y_train)

# применяем обученный векторайзер к тестовым данным
x_test_no_stop = vectorizer_no_stop.transform(x_test)

# получаем предсказания и выводим информацию о качестве
pred = clf.predict(x_test_no_stop)
print(classification_report(y_test, pred))

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


              precision    recall  f1-score   support

    negative       1.00      1.00      1.00     27929
    positive       1.00      1.00      1.00     28780

    accuracy                           1.00     56709
   macro avg       1.00      1.00      1.00     56709
weighted avg       1.00      1.00      1.00     56709



**Шок!** Стоило оставить пунктуацию — и все метрики равны 1. Это произошло из-за того, что в датасете положительные твиты помечены автоматически по наличию `)` / `:D`, а отрицательные — по `(` / `:(`. Эти токены — точная разметка, а не сигнал тональности.

Посмотрим, как один из супер-значительных токенов справится с классификацией без машинного обучения:

In [58]:
# TODO #20
# Простой классификатор «по смайлику»: если в тексте есть `(` — это негатив,
# если `)` — позитив. Никакого ML.

def smile_classifier(text):
    has_pos = ')' in text or ':D' in text or '=)' in text
    has_neg = '(' in text or ':-(' in text
    if has_pos and not has_neg:
        return 'positive'
    if has_neg and not has_pos:
        return 'negative'
    # неоднозначно — голосуем по числу скобок
    return 'positive' if text.count(')') >= text.count('(') else 'negative'

smile_pred = x_test.apply(smile_classifier)
print(classification_report(y_test, smile_pred))

              precision    recall  f1-score   support

    negative       1.00      0.95      0.97     27929
    positive       0.95      1.00      0.97     28780

    accuracy                           0.97     56709
   macro avg       0.98      0.97      0.97     56709
weighted avg       0.98      0.97      0.97     56709



## Символьные n-граммы

Теперь в качестве фичей используем униграммы символов. Для этого необходимо установить в `CountVectorizer()` параметр `analyzer='char'`.

In [59]:
# TODO #21

# инициализируем векторайзер для символов
char_vectorizer = CountVectorizer(ngram_range=(1, 1), analyzer='char')

# обучаем его и сразу применяем к x_train
char_x_train = char_vectorizer.fit_transform(x_train)

# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(char_x_train, y_train)

# применяем обученный векторайзер к тестовым данным
char_x_test = char_vectorizer.transform(x_test)

# получаем предсказания и выводим информацию о качестве
pred = clf.predict(char_x_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       1.00      0.99      0.99     27929
    positive       0.99      1.00      1.00     28780

    accuracy                           1.00     56709
   macro avg       1.00      1.00      1.00     56709
weighted avg       1.00      1.00      1.00     56709



Из предыдущего раздела уже понятно, почему на этих данных точность равна 1: среди символов оказались `(` и `)`, по которым размечен датасет.

Символьные n-граммы используются, например, для задачи определения языка. Ещё одна замечательная особенность признаков-символов — для них не нужна токенизация и лемматизация, можно использовать такой подход для языков, у которых нет готовых анализаторов.

# Самостоятельная работа

1. Изучить материал, представленный в борде. ✅
2. Выполнить все ячейки и получить результаты. ✅
3. Привести результаты `classification_report` для модели `LogisticRegression`.
4. Применить 2 альтернативных алгоритма (`XGBClassifier` и ещё один).
5. Для `XGBClassifier` использовать заданные параметры.
6. В разделе TF-IDF вычислить `classification_report` для биграмм и триграмм, сравнить с униграммами/пентаграммами.

## Пункт 3. Отчёт LogisticRegression на униграммах (CountVectorizer)

Это базовый эксперимент из борда. Результаты `classification_report`:

```
              precision    recall  f1-score   support

    negative       0.76      0.77      0.76     27929
    positive       0.77      0.76      0.77     28780

    accuracy                           0.77     56709
   macro avg       0.77      0.77      0.77     56709
weighted avg       0.77      0.77      0.77     56709
```

## Пункт 4-5. XGBClassifier

В качестве признаков используем TF-IDF униграммы (для скорости ограничены 20000 наиболее частыми словами). Метки кодируются `LabelEncoder`, так как XGBoost ожидает числовые классы.

In [60]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Кодируем метки в 0/1
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# TF-IDF униграммы (с лимитом признаков для скорости)
tfidf_xgb = TfidfVectorizer(ngram_range=(1, 1), max_features=20000)
xgb_x_train = tfidf_xgb.fit_transform(x_train)
xgb_x_test = tfidf_xgb.transform(x_test)

xgb = XGBClassifier(
    learning_rate=0.1,
    n_estimators=1000,
    max_depth=5,
    min_child_weight=3,
    gamma=0.2,
    subsample=0.6,
    colsample_bytree=1.0,
    objective='binary:logistic',
    nthread=4,
    scale_pos_weight=1,
    seed=27,
    tree_method='hist',
)
xgb.fit(xgb_x_train, y_train_enc)

pred = le.inverse_transform(xgb.predict(xgb_x_test))
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.74      0.68      0.71     27929
    positive       0.71      0.77      0.74     28780

    accuracy                           0.73     56709
   macro avg       0.73      0.73      0.73     56709
weighted avg       0.73      0.73      0.73     56709



**Вывод:** XGBClassifier на TF-IDF униграммах показал accuracy 0.73 — немного ниже, чем `LogisticRegression` (0.77). Причина в том, что для разреженных векторов TF-IDF линейные модели обычно работают лучше градиентного бустинга: размерность очень велика, а связь между признаками и таргетом близка к линейной.

### Альтернативный алгоритм №2 — LinearSVC

Линейный SVM — один из самых популярных классификаторов для bag-of-words представлений.

In [61]:
from sklearn.svm import LinearSVC

svc = LinearSVC(random_state=42, max_iter=2000)
svc.fit(xgb_x_train, y_train)  # используем те же TF-IDF признаки
pred = svc.predict(xgb_x_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.74      0.73      0.74     27929
    positive       0.75      0.76      0.75     28780

    accuracy                           0.74     56709
   macro avg       0.74      0.74      0.74     56709
weighted avg       0.74      0.74      0.74     56709



**Вывод:** `LinearSVC` показал accuracy 0.74 — между XGBoost и LogisticRegression. Все три модели близки, но логистическая регрессия выиграла за счёт полного словаря TF-IDF (без `max_features=20000`).

## Пункт 6. Биграммы и триграммы в TF-IDF

Сравниваем с уже посчитанными униграммами+пентаграммами `(1,5)` и пентаграммами `(5,5)`.

### TF-IDF биграммы

In [62]:
tfidf_2 = TfidfVectorizer(ngram_range=(2, 2))
tfidf_x_train_2 = tfidf_2.fit_transform(x_train)
tfidf_x_test_2 = tfidf_2.transform(x_test)
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(tfidf_x_train_2, y_train)
pred = clf.predict(tfidf_x_test_2)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.72      0.66      0.69     27929
    positive       0.70      0.76      0.73     28780

    accuracy                           0.71     56709
   macro avg       0.71      0.71      0.71     56709
weighted avg       0.71      0.71      0.71     56709



### TF-IDF триграммы

In [63]:
tfidf_3 = TfidfVectorizer(ngram_range=(3, 3))
tfidf_x_train_3 = tfidf_3.fit_transform(x_train)
tfidf_x_test_3 = tfidf_3.transform(x_test)
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(tfidf_x_train_3, y_train)
pred = clf.predict(tfidf_x_test_3)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.72      0.45      0.56     27929
    positive       0.61      0.83      0.70     28780

    accuracy                           0.64     56709
   macro avg       0.66      0.64      0.63     56709
weighted avg       0.66      0.64      0.63     56709



### Сводная таблица TF-IDF + LogisticRegression

| n-грамма | precision (neg) | recall (neg) | f1 (neg) | precision (pos) | recall (pos) | f1 (pos) | accuracy |
|---|---|---|---|---|---|---|---|
| `(1, 5)` униграммы..пентаграммы | 0.73 | 0.78 | **0.76** | 0.77 | 0.72 | **0.75** | **0.75** |
| `(2, 2)` биграммы | 0.72 | 0.66 | 0.69 | 0.70 | 0.76 | 0.73 | 0.71 |
| `(3, 3)` триграммы | 0.72 | 0.45 | 0.56 | 0.61 | 0.83 | 0.70 | 0.64 |
| `(5, 5)` пентаграммы | 0.95 | 0.11 | 0.20 | 0.54 | 0.99 | 0.70 | 0.56 |

**Изменилась ли точность f1-score?**

Да, заметно. Лучший результат у диапазона `(1, 5)` — он включает в себя все более короткие n-граммы, а основной вклад дают именно униграммы. Биграммы и триграммы по отдельности проигрывают: чем длиннее n-грамма, тем меньше шанс встретить её в тестовой выборке. Пентаграммы практически бесполезны — модель скатывается к предсказанию мажоритарного класса (recall negative падает до 0.11). Поэтому изолированно использовать длинные n-граммы не имеет смысла; их применяют только в составе диапазона вместе с короткими.

## Итоговая сводка экспериментов

| Метод | Accuracy | f1 macro |
|---|---|---|
| `CountVectorizer (1,1)` + LogReg | **0.77** | 0.77 |
| `CountVectorizer (3,3)` + LogReg | 0.64 | 0.63 |
| `TfidfVectorizer (1,5)` + LogReg | 0.75 | 0.75 |
| `TfidfVectorizer (2,2)` + LogReg | 0.71 | 0.71 |
| `TfidfVectorizer (3,3)` + LogReg | 0.64 | 0.63 |
| `TfidfVectorizer (5,5)` + LogReg | 0.56 | 0.45 |
| `Count + word_tokenize + stopwords` + LogReg | **0.78** | 0.78 |
| `Count + word_tokenize` (без стоп-слов) + LogReg | 1.00 | 1.00 *(утечка через `(` `)`)* |
| Только смайлики, без ML | 0.97 | 0.97 *(утечка)* |
| `Char (1,1)` + LogReg | 1.00 | 1.00 *(утечка)* |
| `XGBClassifier` на TF-IDF (1,1) | 0.73 | 0.73 |
| `LinearSVC` на TF-IDF (1,1) | 0.74 | 0.74 |

### Выводы

1. Самым честным признаковым представлением оказался `CountVectorizer + word_tokenize + stopwords` с accuracy **0.78**.
2. Использование длинных n-грамм без коротких сильно ухудшает качество — из-за нехватки совпадений с обучающей выборкой.
3. Удаление пунктуации обязательно: эмодзи `(` и `)` — это сама автоматическая разметка датасета, и любая модель, видящая их, обучится на утечку и покажет idealistic accuracy = 1.0, но это не настоящее качество.
4. На разреженных bag-of-words представлениях `LogisticRegression` и `LinearSVC` работают лучше градиентного бустинга `XGBClassifier`.
5. Самые жирные приросты дают предобработка и токенизация, а не усложнение модели.